# Lab 3: Search by Meaning

## This lab has 3 steps.

1. Establish an exact semantic-search baseline.
2. Create the cosine DiskANN index.
3. Run approximate search and prove the indexed path.

## Lab Goals

Compare exact similarity search with approximate DiskANN retrieval, and distinguish plausible results from proof that the vector index was used.

## The challenge

Return to **"an elephant is standing on my torso."** Full-text search could
match the words without finding the intended concept. With vectors in place,
we can instead compare the query's meaning with the stored case text.

There are two different search paths. Exact search computes a distance for
every candidate chunk. Approximate search uses an index to find promising
neighbors, trading exact recall for an indexed retrieval strategy.

![Exact kNN evaluates distances across all stored vectors; approximate DiskANN retrieval uses an index to find candidate neighbors, which may differ from the exact top results.](../img/exact-and-approximate.png)

Conceptual comparison only: the points are not measured embeddings or an actual DiskANN graph. Compare relevance and recall, then inspect the actual plan for Vector Index Seek.

The dataset is deliberately small. It can teach query shape and relevance,
but it cannot establish production performance or guarantee approximate recall.

## Starting point

Complete notebook 2 with no pending embeddings. Keep the MSSQL connection on
`Healthcare` and run each cell individually. Each search example makes a
billed query-embedding request using the **same model** as the stored vectors.

These are synthetic cases, not medical advice. The [lab guide](README.md)
explains engine availability and how to handle an unavailable indexed path.


## Step 00: Verify this lab is ready to run

A fresh VM should already have the required database setup. Check that state before creating or calling anything.
This read-only checkpoint reports one ordered Result column and makes no
billed model calls. Every required check must say **Yes**.

A **No** or **Skipped** result produces controlled error 51000 after the
grid. Follow its diagnostic and rerun the checkpoint; the script does not
provision or repair resources for you.


In [ ]:
SET NOCOUNT ON;
DECLARE @Checks TABLE (
    Sort INT IDENTITY PRIMARY KEY, [Check] NVARCHAR(100),
    Passed BIT NOT NULL, Skipped BIT NOT NULL DEFAULT 0, Details NVARCHAR(2048)
);
DECLARE @DatabaseReady BIT = CASE WHEN DB_NAME() = N'Healthcare' THEN 1 ELSE 0 END;
DECLARE @Passed BIT;
INSERT INTO @Checks ([Check], Passed, Details)
VALUES (N'Workshop database', @DatabaseReady, N'Connect the MSSQL kernel to Healthcare, not master.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL AND OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN EXISTS (SELECT 1 FROM dbo.RetrievalChunks)
    AND NOT EXISTS (SELECT 1 FROM dbo.Cases c
      WHERE NOT EXISTS (SELECT 1 FROM dbo.RetrievalChunks ch WHERE ch.CaseId = c.Id)) THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Chunks cover every case', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL AND OBJECT_ID(N'dbo.Cases', N'U') IS NOT NULL) THEN 1 ELSE 0 END,
    N'Run retrieval Lab 02; every source case must have at least one chunk.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN (SELECT COUNT(*) FROM dbo.RetrievalChunks WHERE Embedding IS NOT NULL) >= 100
    AND NOT EXISTS (SELECT 1 FROM dbo.RetrievalChunks WHERE Embedding IS NULL) THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Complete embeddings', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL) THEN 1 ELSE 0 END,
    N'Resume retrieval Lab 02; all chunks need vectors, with at least 100 for the vector index.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (1 = 1)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN EXISTS (SELECT 1 FROM sys.external_models m
    JOIN sys.database_scoped_credentials c ON c.credential_id = m.credential_id
    WHERE m.name = N''embedding_openai_text3_small''
      AND c.credential_identity = N''Managed Identity'') THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'External embedding model', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (1 = 1) THEN 1 ELSE 0 END,
    N'Complete retrieval Lab 01 and inspect the model/credential association.');

SELECT CONCAT([Check], N': ', CASE
    WHEN Skipped = 1 THEN CONCAT(N'Skipped - ', Details)
    WHEN Passed = 1 THEN N'Yes'
    ELSE CONCAT(N'No - ', Details)
END) AS Result
FROM @Checks ORDER BY Sort;
IF EXISTS (SELECT 1 FROM @Checks WHERE Passed = 0)
    THROW 51000, 'Resolve the failed or skipped required checks above.', 1;


### Expected Output
Estimated time: 1-5 seconds.

| Result |
|---|
| Workshop database: Yes |
| Chunks cover every case: Yes |
| Complete embeddings: Yes |
| External embedding model: Yes |


## Step 1: Establish an exact semantic-search baseline

Embed our original idiom with the same model used for the case chunks,
then calculate cosine distance against every stored vector. Smaller distance
means closer meaning. This exact kNN query is our comparison baseline: adding
an index later will not turn `VECTOR_DISTANCE` into an indexed operation.

**Predict:** Which chest-pain cases might move up compared with the original
full-text query? Semantic search can still retrieve irrelevant text, so read
the evidence to assess whether it improves relevance for this query.

> **Inference cost:** One billed query-embedding request. Results are
> model-dependent similarities, not diagnoses or probabilities.

**Learn more:** [VECTOR_DISTANCE](https://learn.microsoft.com/en-us/sql/t-sql/functions/vector-distance-transact-sql?view=sql-server-ver17).


In [ ]:
DECLARE @QueryVector VECTOR(1536) = AI_GENERATE_EMBEDDINGS(
    N'An elephant is standing on my torso' USE MODEL embedding_openai_text3_small
);
SELECT TOP (10) ch.Id AS ChunkId, ch.CaseId, ch.Chunk,
    VECTOR_DISTANCE('cosine', ch.Embedding, @QueryVector) AS Distance
FROM dbo.RetrievalChunks ch
ORDER BY VECTOR_DISTANCE('cosine', ch.Embedding, @QueryVector), ch.Id;


### Expected result
Estimated time: 1-15 seconds.

Up to ten chunks ordered by ascending distance. Review relevance to assess how semantic search complements lexical retrieval for this query.


## Step 2: Create the cosine DiskANN index

Create the cosine DiskANN index over the populated vector column.
It enables the approximate path, but only for a compatible query shape and
metric; index existence by itself is not proof of index usage.

We already checked the clustered key and complete embeddings. Current
vector-index creation needs at least 100 non-NULL vectors. If the selected
engine cannot build the authored index, stop and report that limitation
instead of relabeling exact search as DiskANN.

**Learn more:** [CREATE VECTOR INDEX](https://learn.microsoft.com/en-us/sql/t-sql/statements/create-vector-index-transact-sql?view=sql-server-ver17).


In [ ]:
IF NOT EXISTS (SELECT 1 FROM sys.indexes
    WHERE object_id = OBJECT_ID(N'dbo.RetrievalChunks') AND name = N'IX_RetrievalChunks_Embedding')
    CREATE VECTOR INDEX IX_RetrievalChunks_Embedding
    ON dbo.RetrievalChunks(Embedding) WITH (METRIC = 'cosine', TYPE = 'diskann');


### Expected result
Estimated time: 1-30 seconds.

`IX_RetrievalChunks_Embedding` exists. If this preview feature is unavailable on the selected engine, stop and use the documented exact-search comparison; do not call it a completed DiskANN lab.


### Inspect the result

Inspect the newly created index and its build parameters. Expect one
row for `IX_RetrievalChunks_Embedding` on `dbo.RetrievalChunks`.

This establishes that an index exists. The next exercise is deliberately
separate: it checks whether the search actually uses that index.


In [ ]:
SELECT i.name, v.build_parameters
FROM sys.vector_indexes v JOIN sys.indexes i
    ON i.object_id = v.object_id AND i.index_id = v.index_id
WHERE i.object_id = OBJECT_ID(N'dbo.RetrievalChunks') AND i.name = N'IX_RetrievalChunks_Embedding';


### Expected result
Estimated time: 1-5 seconds.

One vector-index row. This proves existence, not index usage; the next exercise tests usage.


## Step 3: Run approximate search and prove the indexed path

Ask for approximate neighbors with `TOP ... WITH APPROXIMATE` and
`VECTOR_SEARCH`, then inspect the execution-plan XML returned by
`SET STATISTICS XML ON`. Look for **Vector Index Seek** in the search plan.
If only a scan is present, investigate the index and metric before claiming
the DiskANN objective is met.

Compare the top-ten chunk IDs with the exact baseline. Differences can be
part of the approximation; neither plausible rows nor a quick execution
proves indexed retrieval or perfect recall.

> **Inference cost:** One billed query embedding. Keep `r.distance` ascending
> as the approximate query's only sort key.

**Learn more:** [VECTOR_SEARCH](https://learn.microsoft.com/en-us/sql/t-sql/functions/vector-search-transact-sql?view=sql-server-ver17).

**Reflect:** What did the result grid tell you, and what additional evidence
did the execution plan provide?


In [ ]:
SET STATISTICS XML ON;
DECLARE @QueryVector VECTOR(1536) = AI_GENERATE_EMBEDDINGS(
    N'An elephant is standing on my torso' USE MODEL embedding_openai_text3_small
);
SELECT TOP (10) WITH APPROXIMATE ch.Id AS ChunkId, ch.CaseId, ch.Chunk, r.distance
FROM VECTOR_SEARCH(
    TABLE = dbo.RetrievalChunks AS ch, COLUMN = Embedding,
    SIMILAR_TO = @QueryVector, METRIC = 'cosine'
) AS r
ORDER BY r.distance;
SET STATISTICS XML OFF;


### Expected result
Estimated time: 1-15 seconds.

Nearest chunks in ascending distance and execution-plan XML. Confirm Vector Index Seek in the VECTOR_SEARCH plan; if it is missing, the DiskANN exercise is not validated even if rows are returned. WITH APPROXIMATE must keep distance as its only sort key.


## Final Step: Validate this lab is complete

Before moving on, verify the persistent handoffs this notebook owns or depends on.
This read-only checkpoint reports one ordered Result column and makes no
billed model calls. Every required check must say **Yes**.

A **No** or **Skipped** result produces controlled error 51000 after the
grid. Follow its diagnostic and rerun the checkpoint; the script does not
provision or repair resources for you.

This checks persisted state, not live connectivity, search relevance, or index use. Those require inspecting the explicit exercises above.


In [ ]:
SET NOCOUNT ON;
DECLARE @Checks TABLE (
    Sort INT IDENTITY PRIMARY KEY, [Check] NVARCHAR(100),
    Passed BIT NOT NULL, Skipped BIT NOT NULL DEFAULT 0, Details NVARCHAR(2048)
);
DECLARE @DatabaseReady BIT = CASE WHEN DB_NAME() = N'Healthcare' THEN 1 ELSE 0 END;
DECLARE @Passed BIT;
INSERT INTO @Checks ([Check], Passed, Details)
VALUES (N'Workshop database', @DatabaseReady, N'Connect the MSSQL kernel to Healthcare, not master.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN (SELECT COUNT(*) FROM dbo.RetrievalChunks WHERE Embedding IS NOT NULL) >= 100
    AND NOT EXISTS (SELECT 1 FROM dbo.RetrievalChunks WHERE Embedding IS NULL) THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'Complete embeddings', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (OBJECT_ID(N'dbo.RetrievalChunks', N'U') IS NOT NULL) THEN 1 ELSE 0 END,
    N'Resume retrieval Lab 02; all chunks need vectors, with at least 100 for the vector index.');

SET @Passed = 0;
IF @DatabaseReady = 1 AND (1 = 1)
    EXEC sys.sp_executesql
        N'SELECT @Ok = CASE WHEN EXISTS (SELECT 1 FROM sys.vector_indexes v JOIN sys.indexes i
    ON i.object_id = v.object_id AND i.index_id = v.index_id
    WHERE i.object_id = OBJECT_ID(N''dbo.RetrievalChunks'') AND i.name = N''IX_RetrievalChunks_Embedding'') THEN 1 ELSE 0 END;',
        N'@Ok BIT OUTPUT', @Ok = @Passed OUTPUT;
INSERT INTO @Checks ([Check], Passed, Skipped, Details)
VALUES (N'retrieval vector index', @Passed,
    CASE WHEN @DatabaseReady = 0 OR NOT (1 = 1) THEN 1 ELSE 0 END,
    N'Complete retrieval Lab 03 and verify the approximate-search execution plan.');

SELECT CONCAT([Check], N': ', CASE
    WHEN Skipped = 1 THEN CONCAT(N'Skipped - ', Details)
    WHEN Passed = 1 THEN N'Yes'
    ELSE CONCAT(N'No - ', Details)
END) AS Result
FROM @Checks ORDER BY Sort;
IF EXISTS (SELECT 1 FROM @Checks WHERE Passed = 0)
    THROW 51000, 'Resolve the failed or skipped required checks above.', 1;


### Expected Output
Estimated time: 1-5 seconds.

| Result |
|---|
| Workshop database: Yes |
| Complete embeddings: Yes |
| retrieval vector index: Yes |


Previous: [Notebook 2](02-case-notes-to-vectors.ipynb) | [Session guide](README.md) | Next: [Notebook 4](04-narrative-to-results.ipynb)
